# 01 · Crossing the Bifrost – a first call
The gateway exposes an **OpenAI-compatible** endpoint. Clients only know the gateway URL and *their* subscription key – never the Foundry endpoint or credentials (the gateway uses its managed identity).

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

r = ask("In one sentence, who is Heimdall in Marvel?", user="thor")
print(r['answer'])
print({k: r[k] for k in ('user', 'tier', 'model', 'reason', 'remaining_tokens')})

Response headers `x-heimdal-*` are added by the gateway so every call explains *who* asked and *which model* answered.

### Responses API and embeddings
`POST /responses` goes through the **same router** as chat completions (agents and newer SDKs use it). `POST /embeddings` skips the router: `heimdal-auto` maps to the embedding deployment (`model-embedding`).

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

raw = client('thor').responses.with_raw_response.create(model='heimdal-auto', input='Name the guardian of the Bifrost in three words.')
print(raw.parse().output_text)
print({h: raw.headers.get(h) for h in ('x-heimdal-tier', 'x-heimdal-model', 'x-heimdal-route-reason')})

emb = client('thor').embeddings.with_raw_response.create(model='heimdal-auto', input=['Bifrost', 'Mjolnir'])
print(len(emb.parse().data), 'vectors of', len(emb.parse().data[0].embedding), 'dims via', emb.headers.get('x-heimdal-model'))

### Optional: a model from your own machine (Ollama via ngrok)
Deployed with `HEIMDAL_ENABLE_OLLAMA=true`, the gateway also knows the local models in `HEIMDAL_OLLAMA_MODELS`. Same endpoint, same key, same token limit: APIM forwards the call through the ngrok tunnel (basic auth) to Ollama on the demo machine.

In [ ]:
import os, sys; sys.path.insert(0, '.')
from heimdal_helpers import *

local = [m for m in os.getenv('HEIMDAL_OLLAMA_MODELS', '').split(',') if m]
if not local:
    print('Local Ollama is not enabled (see README -> Local Ollama via ngrok).')
else:
    r = ask('In one sentence, who is Heimdall in Marvel?', user='thor', model=local[0])
    print(r['answer'])
    print({k: r[k] for k in ('model', 'tier', 'reason', 'remaining_tokens')})